# Spline Activations Trade Parameters for Compute in KANs

**Paper:** [https://arxiv.org/abs/2404.19756](https://arxiv.org/abs/2404.19756)  
**Authors:** Ziming Liu, Yixuan Wang, Sachin Vaidya, Fabian Ruehle, James Halverson, Marin Soljačić, Thomas Y. Hou, Max Tegmark  
**Repository:** [https://github.com/KindXiaoming/pykan](https://github.com/KindXiaoming/pykan)  
**License:** MIT  

---

*Reproduction generated by Vivory Research — runs on free-tier hardware (Kaggle T4 / Oracle CPU / GitHub Actions).*
*Produced: 2026-05-21 06:55 UTC*


## 1. Setup

Install dependencies from the paper's `requirements.txt`. Some packages may need GPU-specific wheels — adjust for your Colab/Kaggle runtime.

In [ ]:
!pip install --quiet --upgrade pip
!pip install --quiet matplotlib==3.6.2 numpy==1.24.4 scikit_learn==1.1.3 setuptools==65.5.0 sympy==1.11.1 torch==2.2.2 tqdm==4.66.2 pandas==2.0.1 seaborn pyyaml


## 2. Repository

Clone the reference implementation.

In [ ]:
!git clone --depth 1 https://github.com/KindXiaoming/pykan
%cd pykan
!ls -la


## 3. Dataset

Download the dataset. Replace this cell with the dataset-specific loading code from the repository's README or `scripts/download_data.sh`.

In [ ]:
# TODO: Replace with dataset-specific download/load code.
# Check the repo README for instructions — common patterns:
#   bash scripts/download_data.sh
#   python -m src.data.download
#   from datasets import load_dataset; ds = load_dataset("name")
print("Dataset placeholder — fill in from repo README.")


## 4. Configuration

Core hyperparameters. Consider reducing epochs/batch size to fit free-tier GPU limits (Kaggle T4: 16GB VRAM, 30h/week; Colab: variable).

In [ ]:
import os, json, random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Reduced for free-tier — adjust if you have more GPU budget.
CONFIG = {
    "seed": SEED,
    "max_epochs": 1,
    "batch_size": 16,
    "learning_rate": 1e-4,
    "subset_fraction": 0.1,  # use 10% of data for quick reproduction
}
print(json.dumps(CONFIG, indent=2))


## 5+6. Paper-aware evaluation (auto-generated)

The cell below was generated by Vivory's reproduction agent (Opus 4.7) from the paper's abstract, body, repo README, and claimed_metrics. It performs real measurement on a small subset and writes the result to `/kaggle/working/metrics.json` for the runner to ingest.

In [ ]:
# KAN: Kolmogorov-Arnold Networks (arXiv:2404.19756) — PDE-solving reproduction.
# Paper claim: a small 2-layer width-10 KAN solves a 2D Poisson PDE more
# accurately AND with far fewer parameters than a larger MLP (PINN setup,
# ~1e-7 vs ~1e-5 MSE, ~1e2 vs ~1e4 params). We train both with a real
# physics-informed loss and measure test MSE vs the analytic solution plus
# trainable parameter counts. All numbers below are measured, not assumed.

# pykan is on PyPI; install with --no-deps to avoid churning Kaggle's torch/numpy.
!pip install -q pykan --no-deps
try:
    import kan  # noqa: F401
except Exception:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                    "git+https://github.com/KindXiaoming/pykan.git"], check=False)

import os, json, math, time

os.makedirs("/kaggle/working", exist_ok=True)
METRICS_PATH = "/kaggle/working/metrics.json"

def _write(d):
    with open(METRICS_PATH, "w") as f:
        json.dump(d, f, indent=2)
    print(d)

try:
    import torch
    import torch.nn as nn
    from torch import autograd
    from kan import KAN

    device = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(0)
    PI = math.pi

    # --- 2D Poisson on [-1,1]^2 : lap(u)=f, u=0 on boundary ---
    # analytic solution u* = sin(pi x) sin(pi y)  =>  f = -2 pi^2 u*
    def u_true(xy):
        return torch.sin(PI * xy[:, :1]) * torch.sin(PI * xy[:, 1:2])

    def source(xy):
        return -2.0 * PI * PI * torch.sin(PI * xy[:, :1]) * torch.sin(PI * xy[:, 1:2])

    def sample_interior(n):
        return torch.rand(n, 2, device=device) * 2.0 - 1.0

    def sample_boundary(n):
        m = n // 4
        e = torch.rand(m, device=device) * 2.0 - 1.0
        o = torch.ones(m, device=device)
        return torch.cat([
            torch.stack([e, -o], 1), torch.stack([e, o], 1),
            torch.stack([-o, e], 1), torch.stack([o, e], 1),
        ], 0)

    X_int = sample_interior(1000)
    X_bc = sample_boundary(400)
    f_int = source(X_int)

    def _as2d(u):
        return u if u.dim() == 2 else u.unsqueeze(-1)

    def laplacian(model, x):
        x = x.clone().requires_grad_(True)
        u = _as2d(model(x))
        g = autograd.grad(u.sum(), x, create_graph=True)[0]
        uxx = autograd.grad(g[:, :1].sum(), x, create_graph=True)[0][:, :1]
        uyy = autograd.grad(g[:, 1:2].sum(), x, create_graph=True)[0][:, 1:2]
        return uxx + uyy

    def pde_loss(model):
        res = laplacian(model, X_int) - f_int
        ub = _as2d(model(X_bc))
        return (res ** 2).mean() + (ub ** 2).mean()

    def train_lbfgs(model, outer, budget_s):
        params = [p for p in model.parameters() if p.requires_grad]
        if len(params) == 0:
            return
        opt = torch.optim.LBFGS(params, lr=1.0, max_iter=15, history_size=10,
                                line_search_fn="strong_wolfe")
        t0 = time.time()
        for _ in range(outer):
            def closure():
                opt.zero_grad()
                l = pde_loss(model)
                l.backward()
                return l
            opt.step(closure)
            if time.time() - t0 > budget_s:
                break

    # --- fixed evaluation grid where the exact solution is known ---
    gg = torch.linspace(-1, 1, 50, device=device)
    gx, gy = torch.meshgrid(gg, gg, indexing="ij")
    X_test = torch.stack([gx.reshape(-1), gy.reshape(-1)], 1)
    U_test = u_true(X_test)

    @torch.no_grad()
    def test_mse(model):
        return float(((_as2d(model(X_test)) - U_test) ** 2).mean().item())

    def n_params(model):
        return float(sum(p.numel() for p in model.parameters() if p.requires_grad))

    # --- KAN: 2-layer width-10, learnable B-spline activations on edges ---
    try:
        kan_model = KAN(width=[2, 10, 1], grid=5, k=3, seed=1,
                        device=device, symbolic_enabled=False)
    except TypeError:
        kan_model = KAN(width=[2, 10, 1], grid=5, k=3, seed=1).to(device)
    try:
        kan_model.speed()  # disable slow symbolic branch if API supports it
    except Exception:
        pass

    train_lbfgs(kan_model, outer=30, budget_s=360)
    # grid-extension refinement (the paper's accuracy-boosting technique)
    try:
        refined = kan_model.refine(10)
        if refined is not None:
            kan_model = refined
        train_lbfgs(kan_model, outer=20, budget_s=200)
    except Exception:
        pass

    mse_kan = test_mse(kan_model)
    params_kan = n_params(kan_model)

    # --- MLP: deeper/wider, fixed tanh activations (~1e4 params) ---
    class MLP(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(2, 100), nn.Tanh(),
                nn.Linear(100, 100), nn.Tanh(),
                nn.Linear(100, 1))
        def forward(self, x):
            return self.net(x)

    mlp = MLP().to(device)
    train_lbfgs(mlp, outer=50, budget_s=300)
    mse_mlp = test_mse(mlp)
    params_mlp = n_params(mlp)

    measured = {
        "mse_pde_kan": mse_kan,
        "mse_pde_mlp": mse_mlp,
        "parameter_count_kan": params_kan,
        "parameter_count_mlp": params_mlp,
    }
    # coerce any non-finite floats to None so metrics.json stays valid
    measured = {k: (v if (isinstance(v, (int, float)) and math.isfinite(v)) else None)
                for k, v in measured.items()}
    _write(measured)

except Exception as e:
    # honest, diagnosable failure instead of crashing the runner
    _write({"infrastructure_error": str(e)[:300]})

## Appendix — Reproduction policy

This notebook runs on **free-tier hardware only**:

- **Kaggle Notebooks** — T4 GPU, 30h/week quota
- **Oracle Cloud** — ARM 4-core CPU, no GPU
- **GitHub Actions** — 2-core CPU, no GPU, 6h timeout
- **Colab** — variable T4/V100, 12h sessions (manual only)

If the full experiment exceeds these limits, reduce `max_epochs` / `subset_fraction` in the config cell and note the delta in the reproduction report.
